# CPPWRF ile hava tahmini (Colab)

Bu notebook, NOAA'nın en son **GFS** koşusundan başlayıp **CPPWRF** (WRF'nin C++/CUDA sürümü) ile bölgesel bir hava
tahmini yapar, sonra haritaları ve şehir tahminlerini çizer:

1. GFS (0,25°) dosyaları indirilir (NOAA açık verisi).
2. **WPS** (geogrid, ungrib, metgrid) modeli hazırlar. WPS burada derlenir, WRF'nin tamamının derlenmesi gerekmez.
3. `cppwrf_real` (WRF real.exe) başlangıç ve sınır dosyalarını yazar.
4. `cppwrf` (WRF wrf.exe) tahmini **GPU**'da koşturur.
5. Saatlik haritalar (sıcaklık, basınç, yağış, rüzgâr), animasyon ve şehir tahminleri çizilir.

Varsayılan alan **Türkiye**: 220 x 130 nokta, 12 km çözünürlük, 24 saat. Fizik paketi WRF'nin `CONUS` paketidir.

**Önce:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **GPU** (T4 yeter). Sonra hücreleri sırayla çalıştır.
İlk kurulum (derleme) ücretsiz T4'te 20-30 dakika sürer, bu bir kez yapılır. Aynı oturumda yeni tahmin için yalnızca
5-7. hücreleri yeniden çalıştırman yeterli.

Bu bir deneme / eğitim aracıdır, resmi tahmin değildir. Resmi tahmin için MGM'ye bak.

In [ ]:
#@title 1) Ayarlar
LAT = 39.0  #@param {type:"number"}
LON = 35.0  #@param {type:"number"}
NX = 220  #@param {type:"integer"}
NY = 130  #@param {type:"integer"}
DX_KM = 12  #@param {type:"integer"}
HOURS = 24  #@param {type:"integer"}
MODE = "gpu-fp32"  #@param ["gpu-fp32", "gpu-fp64"]
CYCLE = ""  #@param {type:"string"}
#@markdown `CYCLE` boşsa en son GFS koşusu kullanılır (örnek: `2026111706`). `HOURS` 3'ün katı olmalı.
#@markdown Merkez `LAT`/`LON` ve `NX`/`NY`/`DX_KM` alanı belirler: 220 x 12 km = 2640 km doğu-batı.
assert HOURS % 3 == 0, "HOURS 3'ün katı olmalı"
with open("/content/settings.env", "w") as f:
    f.write(f"LAT={LAT}\nLON={LON}\nNX={NX}\nNY={NY}\nDX={DX_KM * 1000}\nHOURS={HOURS}\nMODE={MODE}\nCYCLE={CYCLE}\n")
print(open("/content/settings.env").read())

In [ ]:
%%bash
# 2) GPU ve paketler
nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
apt-get -qq update > /dev/null && apt-get -qq install -y libnetcdf-dev libnetcdff-dev ninja-build csh m4 > /dev/null && echo "apt ok"
pip -q install netCDF4 cartopy && echo "pip ok"
nproc; df -h /content | tail -1

In [ ]:
%%bash
# 3) CPPWRF: kaynak ve GPU derlemesi (ücretsiz T4'te 15-25 dk)
cd /content
[ -d CPPWRF ] || git clone -q https://github.com/SamurayBalgami/CPPWRF.git
cd CPPWRF && git log -1 --format='CPPWRF %h %s'
P=OFF; grep -q 'MODE=gpu-fp64' /content/settings.env && P=ON
B=build_$P
if [ ! -x $B/src/init_real/cppwrf_real ] || [ ! -x $B/cppwrf ]; then
  ARCH=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
  t0=$SECONDS
  cmake -S . -B $B -G Ninja -DCMAKE_BUILD_TYPE=Release -DCPPWRF_ENABLE_CUDA=ON -DCPPWRF_DOUBLE=$P -DCMAKE_CUDA_ARCHITECTURES=$ARCH > $B.cfg.log 2>&1 || { tail -30 $B.cfg.log; exit 1; }
  ninja -C $B -j $(nproc) cppwrf cppwrf_real > $B.log 2>&1 || { grep -m10 -E 'error' $B.log; exit 1; }
  echo "derleme $((SECONDS - t0)) s"
fi
ls -la $B/cppwrf $B/src/init_real/cppwrf_real

In [ ]:
%%bash
# 4) WRF v4.8.0 kaynağı (fizik tabloları; yalnızca G/Ç kütüphaneleri derlenir), WPS 4.6.0, WPS coğrafya verisi
mkdir -p /content/fc && cd /content/fc
export NETCDF=/content/fc/netcdf WRFIO_NCD_LARGE_FILE_SUPPORT=1
mkdir -p netcdf; ln -sfn /usr/bin netcdf/bin; ln -sfn /usr/include netcdf/include; ln -sfn /usr/lib/x86_64-linux-gnu netcdf/lib
t0=$SECONDS
if [ ! -f WRF/external/io_netcdf/libwrfio_nf.a ]; then
  mkdir -p WRF && curl -sfL --retry 3 https://github.com/wrf-model/WRF/releases/download/v4.8.0/v4.8.0.tar.gz | tar xzf - -C WRF --strip-components=1
  cd WRF && printf '32\n1\n' | ./configure > configure.log 2>&1 && ./compile -j $(nproc) io > io.log 2>&1; cd ..
  ls WRF/external/io_netcdf/libwrfio_nf.a || { tail -30 WRF/io.log; exit 1; }
fi
echo "WRF G/Ç $((SECONDS - t0)) s"; t0=$SECONDS
if [ ! -x WPS-4.6.0/metgrid/src/metgrid.exe ]; then
  curl -sfL --retry 3 https://github.com/wrf-model/WPS/archive/refs/tags/v4.6.0.tar.gz | tar xzf -
  cd WPS-4.6.0 && export WRF_DIR=/content/fc/WRF && printf '1\n' | ./configure --build-grib2-libs > configure.log 2>&1 && ./compile > compile.log 2>&1
  # metgrid at -O0: built at -O by newer gfortran it can write met_em files without fields
  sed -i 's/^FFLAGS\( *= *\$(FORMAT_FREE)\) -O /FFLAGS\1 -O0 /; s/^F77FLAGS\( *= *\$(FORMAT_FIXED)\) -O /F77FLAGS\1 -O0 /' configure.wps
  rm -f metgrid/src/*.o metgrid/src/*.mod metgrid/src/metgrid.exe && ./compile metgrid > compile_metgrid.log 2>&1; cd ..
fi
ls WPS-4.6.0/geogrid/src/geogrid.exe WPS-4.6.0/ungrib/src/ungrib.exe WPS-4.6.0/metgrid/src/metgrid.exe || { tail -30 WPS-4.6.0/compile.log; exit 1; }
echo "WPS $((SECONDS - t0)) s"
if [ ! -d geog/WPS_GEOG_LOW_RES ]; then
  mkdir -p geog && curl -sfL --retry 3 https://www2.mmm.ucar.edu/wrf/src/wps_files/geog_low_res_mandatory.tar.gz | tar xzf - -C geog
fi
ls geog

In [ ]:
%%bash
# 5) Tahmin: GFS indir -> WPS -> cppwrf_real -> cppwrf -> haritalar
set -a; . /content/settings.env; set +a
P=OFF; [ "$MODE" = gpu-fp64 ] && P=ON
cd /content/CPPWRF
rm -rf /content/work/maps
WPS_DIR=/content/fc/WPS-4.6.0 WRF_DIR=/content/fc/WRF GEOG_DIR=/content/fc/geog/WPS_GEOG_LOW_RES CPPWRF_BUILD=build_$P \
  bash examples/forecast/forecast.sh /content/work

In [ ]:
# 6) Animasyon ve şehir tahminleri
from IPython.display import Image, display
import glob
display(Image(filename="/content/work/maps/forecast.gif"))
display(Image(filename="/content/work/maps/cities.png"))

In [ ]:
#@title 7) Tek bir saatin haritası
SAAT = 24  #@param {type:"integer"}
from IPython.display import Image, display
display(Image(filename=f"/content/work/maps/map_{SAAT:02d}.png"))

In [ ]:
# 8) Şehir tahminleri (tablo)
print(open("/content/work/maps/cities.txt", encoding="utf-8").read())

In [ ]:
%%bash
# 9) Sonuçları indir: haritalar ve wrfout dosyaları tek zip (Dosyalar panelinden /content/tahmin.zip)
cd /content/work && rm -f /content/tahmin.zip
zip -qr /content/tahmin.zip maps run/wrfout_d01_* run/namelist.input && ls -la /content/tahmin.zip